# Prepare cluster annotations for analysis

This notebook reads a YOLO dataset folder and a CSV with an `image_name` column, joins them on image name, adds `cluster_annotation`, removes `microglia_to_annotate_cluster`, and writes a new CSV.

In [11]:
from pathlib import Path
import pandas as pd
import yaml

In [12]:
def _load_yolo_names(dataset_dir: Path):
    yaml_candidates = list(dataset_dir.glob('*.yaml')) + list(dataset_dir.glob('*.yml'))
    if not yaml_candidates:
        raise FileNotFoundError(f'No YAML file found in {dataset_dir}')

    yaml_path = yaml_candidates[0]
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

    if 'names' not in data:
        raise KeyError(f"'names' not found in {yaml_path}")

    names = data['names']
    if isinstance(names, list):
        return {i: n for i, n in enumerate(names)}
    if isinstance(names, dict):
        return {int(k): v for k, v in names.items()}

    raise TypeError(f"Unsupported YOLO 'names' format in {yaml_path}")


def _build_image_annotation_map(dataset_dir: Path):
    names_map = _load_yolo_names(dataset_dir)
    labels_root = dataset_dir / 'labels'
    if not labels_root.exists():
        raise FileNotFoundError(f"Expected labels directory at {labels_root}")

    label_files = list(labels_root.rglob('*.txt'))
    if not label_files:
        raise FileNotFoundError(f'No label txt files found under {labels_root}')

    image_to_annotation = {}
    ambiguous_images = []

    for label_file in label_files:
        class_ids = []
        with open(label_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls_id = int(float(line.split()[0]))
                class_ids.append(cls_id)

        if not class_ids:
            continue

        unique_ids = sorted(set(class_ids))
        annotations = [names_map.get(cid, f'class_{cid}') for cid in unique_ids]

        if len(annotations) > 1:
            ambiguous_images.append(label_file.stem)

        image_to_annotation[label_file.stem] = '|'.join(annotations)

    if ambiguous_images:
        print(f'Warning: {len(ambiguous_images)} images had multiple annotations; merged with |')

    return image_to_annotation


def prepare_cluster_annotations(dataset_dir, csv_path=None, output_csv_path=None):
    dataset_dir = Path(dataset_dir)
    if csv_path is None:
        csv_files = sorted(dataset_dir.glob('*.csv'))
        if len(csv_files) != 1:
            raise ValueError(f'Expected exactly one CSV in {dataset_dir}, found {len(csv_files)}')
        csv_path = csv_files[0]
    else:
        csv_path = Path(csv_path)

    if output_csv_path is None:
        output_csv_path = csv_path.with_name(f'{csv_path.stem}_with_cluster_annotation.csv')
    else:
        output_csv_path = Path(output_csv_path)

    df = pd.read_csv(csv_path)
    if 'image_name' not in df.columns:
        raise KeyError("Input CSV must contain an 'image_name' column")

    if not df['image_name'].is_unique:
        dupes = df.loc[df['image_name'].duplicated(keep=False), 'image_name'].head(10).tolist()
        print(f'image_name is not unique in CSV. Keeping first entry per image_name. Examples: {dupes}')
        before_dedup = len(df)
        df = df.drop_duplicates(subset=['image_name'], keep='first').copy()
        print(f'Rows removed due to duplicate image_name: {before_dedup - len(df)}')

    image_to_annotation = _build_image_annotation_map(dataset_dir)

    df = df.copy()
    image_stems = df['image_name'].astype(str).map(lambda x: Path(x).stem)
    df['cluster_annotation'] = image_stems.map(image_to_annotation)

    before = len(df)
    df = df[df['cluster_annotation'].notna()].copy()
    removed_missing = before - len(df)

    before_microglia_filter = len(df)
    df = df[df['cluster_annotation'] != 'microglia_to_annotate_cluster'].copy()
    removed_microglia = before_microglia_filter - len(df)

    df.to_csv(output_csv_path, index=False)

    print(f'Input CSV: {csv_path}')
    print(f'Output CSV: {output_csv_path}')
    print(f'Rows removed (missing cluster_annotation): {removed_missing}')
    print(f'Rows removed (microglia_to_annotate_cluster): {removed_microglia}')

    return df

In [14]:

dataset_dir = r"C:\Users\chris\Desktop\University\Thesis\cluster_labels"


prepare_cluster_annotations(
    dataset_dir=dataset_dir,
    output_csv_path=r'cluster_annotations_prepared_for_analysis.csv'
)

image_name is not unique in CSV. Keeping first entry per image_name. Examples: ['tile_x74240_y18944_gray_matter', 'tile_x74240_y18944_gray_matter', 'tile_x36864_y50688_gray_matter', 'tile_x36864_y50688_gray_matter']
Rows removed due to duplicate image_name: 2
Input CSV: C:\Users\chris\Desktop\University\Thesis\cluster_labels\cluster_label_cell_selection_top160.csv
Output CSV: cluster_annotations_prepared_for_analysis.csv
Rows removed (missing cluster_annotation): 22
Rows removed (microglia_to_annotate_cluster): 39


,rank,global_cell_id,scan_name,image_name,xmin,ymin,xmax,ymax,skeleton_length,num_junctions,...,sholl_min_radius,sholl_peak_radius,sholl_max_radius,sholl_peak,sholl_sum,diversity_score,tsne_x,tsne_y,x_row_position,cluster_annotation
0,1,tile_x61440_y38400_gray_matter_cell_1,2017-026_Iba1_Temporal_pole_Batch_8,tile_x61440_y38400_gray_matter,21.504862,19.254723,61.95919,56.013023,0,0,...,0,0,0,0,0,96.995402,-94.085098,-23.581824,48725,Amoeboid
1,2,tile_x35328_y19456_gray_matter_cell_2,2016-020_Iba1_Temporal_pole_Batch_8,tile_x35328_y19456_gray_matter,21.872040,30.620636,185.96805,293.181700,646,17,...,33,62,243,17,646,183.762985,86.926498,8.098445,19874,Hyper-ramified
2,3,tile_x67072_y7680_gray_matter_cell_0,2017-014_Iba1_Temporal_pole_Batch_6,tile_x67072_y7680_gray_matter,407.937260,347.055660,490.75060,508.904800,103,1,...,27,45,65,10,103,111.740791,-14.499291,54.989955,39743,Reactive
5,6,tile_x58368_y24064_gray_matter_cell_0,2017-063_Iba1_Temporal_pole_Batch_8,tile_x58368_y24064_gray_matter,381.136960,245.936500,414.31458,273.659820,1,0,...,13,13,13,1,1,62.750992,-78.975544,37.322926,66008,Amoeboid
9,10,tile_x12800_y10240_gray_matter_cell_2,2017-014_Iba1_Temporal_pole_Batch_6,tile_x12800_y10240_gray_matter,302.167000,314.905850,388.56530,401.507100,100,2,...,13,38,48,8,100,40.230249,-36.699482,-50.213520,32099,Reactive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,152,tile_x59904_y20480_gray_matter_cell_0,2017-160_Iba1_Temporal_pole_Batch_6,tile_x59904_y20480_gray_matter,125.116090,1.067184,168.02030,74.667816,26,0,...,16,24,37,7,26,7.658710,-20.321072,43.298106,90656,Reactive
153,154,tile_x48128_y40960_gray_matter_cell_4,2017-160_Iba1_Temporal_pole_Batch_6,tile_x48128_y40960_gray_matter,340.114260,296.986700,393.09863,379.085820,50,1,...,14,15,50,4,50,7.622900,-22.368086,5.760119,88960,Reactive
156,157,tile_x12800_y36352_gray_matter_cell_1,2017-014_Iba1_Temporal_pole_Batch_6,tile_x12800_y36352_gray_matter,204.992360,362.813870,343.77510,479.654080,262,6,...,9,15,103,6,262,7.545957,41.750900,-18.346213,32149,Amoeboid
157,158,tile_x22528_y22528_gray_matter_cell_0,2016-020_Iba1_Temporal_pole_Batch_8,tile_x22528_y22528_gray_matter,323.475460,141.534850,508.70557,322.159030,305,6,...,10,20,153,9,305,7.524208,49.479698,12.031498,17457,Reactive
